> **Experiment Motivation:** Since Isolation Forest achieved poor detection performance, the next experiment evaluates **Local Outlier Factor (LOF)**. Due to its high computational complexity, the algorithm is applied under computational constraints.

In [1]:
# Import the required libraries and utility functions
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor

from src.utils import calculate_age, calculate_distance_km, transform_cyclic_hour

In [2]:
# Load the datasets
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Combine both datasets into a single DataFrame
df_raw = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# Separate the target variable
y_true = df_raw['is_fraud']
X_raw = df_raw.drop(columns=['is_fraud'])

print("X_raw dimensions:", X_raw.shape)
print("y_true dimensions:", y_true.shape)

X_raw dimensions: (1852394, 22)
y_true dimensions: (1852394,)


In [3]:
# Apply Frequency Encoding using global category frequencies
cols_freq = ["category", "job", "state", "merchant"]

cols_to_drop = [
# Encoded features
    "category",
    "job",
    "state",
    "merchant",
# Processed features
    "trans_date_trans_time",
    "lat",
    "long",
    "merch_lat",
    "merch_long",
    "dob",
# Irrelevant features
    "street",
    "city",
    "zip",
    "unix_time",
    "first",
    "last",
    "cc_num",
    "trans_num",
    "Unnamed: 0",
]

In [4]:
# Apply the same preprocessing pipeline used previously
X_raw['trans_date_trans_time'] = pd.to_datetime(X_raw['trans_date_trans_time'])

# Extract the transaction hour
horas = X_raw['trans_date_trans_time'].dt.hour

# Encode the transaction hour as cyclic features
(
    X_raw["hour_sin"],
    X_raw["hour_cos"],
) = transform_cyclic_hour(X_raw)

# Generate the 'age' and 'distance_km' features and encode 'gender'
X_raw["distance_km"] = calculate_distance_km(X_raw)
X_raw["age"] = calculate_age(X_raw)
X_raw['gender'] = X_raw['gender'].map({'F': 0, 'M': 1})

for col in cols_freq:
# Normalize frequencies to values between 0 and 1
    freq_map = X_raw[col].value_counts(normalize=True)

# Create the encoded feature
    X_raw[f"{col}_freq"] = X_raw[col].map(freq_map)

# Apply log1p to reduce skewness and compress extreme transaction amounts
X_raw["amt"] = np.log1p(X_raw["amt"])

# Drop unnecessary columns
X_raw = X_raw.drop(columns=cols_to_drop, errors="ignore")

# Inspect the resulting columns
X_raw.columns

Index(['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km',
       'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq'],
      dtype='str')

In [5]:
# Configure and run the Local Outlier Factor model
contamination_rate = (y_true == 1).sum() / len(y_true)

print(f"Exact contamination rate: {contamination_rate:.6f}")

# Initialize the LOF model
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination_rate,
    n_jobs=-1
)

# Fit the model and predict anomalies across the full dataset
preds_lof_total = lof.fit_predict(X_raw)

# Map predictions:
# -1 → 1 (Anomaly/Fraud)
#  1 → 0 (Normal)
preds_lof_total_binary = [1 if p == -1 else 0 for p in preds_lof_total]

# Evaluate the model's fraud detection performance
total_frauds = (y_true == 1).sum()
captured_frauds = sum(1 for p, r in zip(preds_lof_total_binary, y_true) if p == 1 and r == 1)

print(f"Total fraud cases:          {total_frauds:,}")
print(f"Frauds detected by LOF:       {captured_frauds:,}")
print(f"Fraud detection rate (Recall):        {(captured_frauds / total_frauds) * 100:.2f}%")

Exact contamination rate: 0.005210
Total fraud cases:          9,651
Frauds detected by LOF:       2,058
Fraud detection rate (Recall):        21.32%


> **Final Conclusion:** In financial fraud detection with historical labels, purely unsupervised methods show limited performance because they identify statistical anomalies rather than deceptive behaviors that intentionally mimic legitimate transactions. This experiment reinforces why supervised models are the preferred approach for production-grade fraud detection systems.
